In [7]:
import torch
from torch import nn
import torch.nn.functional as F
import math

##This is a Transformer decoder block stack that produces text from context-aware embeddings

The decoder generates text one token at a time by:

looking at previous generated tokens,
looking at encoder output,
processing the information,
predicting the next token.

##THE FLOW:


```
DECODER FLOW :

Decoder Input Embeddings
        ↓
Masked Self-Attention
(looks only at previous tokens)
        ↓
Norm + Residual
        ↓
Cross Attention
(decoder attends to encoder output)
        ↓
Norm + Residual
        ↓
FeedForward Network
(process each token independently)
        ↓
Norm + Residual
        ↓
Repeat × N layers
        ↓
Final Decoder Representations
        ↓
Linear Layer + Softmax
        ↓
Next Token Prediction
```



In [8]:
class TransformerDecoderBlock(nn.Module):
    def __init__(
        self,
        model_dim,
        hidden_dim,
        num_heads,
        dropout
    ):
        super().__init__()

        # masked self-attention
        self.self_attention = MultiHeadSelfAttention(
            model_dim,
            num_heads
        )

        self.norm1 = LayerNorm(model_dim)
        self.dropout1 = nn.Dropout(dropout)

        # encoder-decoder attention
        self.cross_attention = MultiHeadCrossAttention(
            model_dim,
            num_heads
        )

        self.norm2 = LayerNorm(model_dim)
        self.dropout2 = nn.Dropout(dropout)

        # feed forward network
        self.ffn = FeedForwardNetwork(
            model_dim,
            hidden_dim,
            dropout
        )

        self.norm3 = LayerNorm(model_dim)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        encoder_output,
        decoder_input,
        decoder_mask=None
    ):
        # ---- Masked Self Attention ----
        residual = decoder_input

        decoder_input = self.self_attention(
            decoder_input,
            mask=decoder_mask
        )

        decoder_input = self.dropout1(decoder_input)

        decoder_input = self.norm1(
            decoder_input + residual
        )

        # ---- Cross Attention ----
        residual = decoder_input

        decoder_input = self.cross_attention(
            encoder_output,
            decoder_input
        )

        decoder_input = self.dropout2(decoder_input)

        decoder_input = self.norm2(
            decoder_input + residual
        )

        # ---- Feed Forward Network ----
        residual = decoder_input

        decoder_input = self.ffn(decoder_input)

        decoder_input = self.dropout3(decoder_input)

        decoder_input = self.norm3(
            decoder_input + residual
        )

        return decoder_input


class TransformerDecoder(nn.Module):
    def __init__(
        self,
        model_dim,
        hidden_dim,
        num_heads,
        dropout,
        num_layers #Repeat N Times
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderBlock(
                model_dim,
                hidden_dim,
                num_heads,
                dropout
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        encoder_output,
        decoder_input,
        decoder_mask=None
    ):
        for layer in self.layers:
            decoder_input = layer(
                encoder_output,
                decoder_input,
                decoder_mask
            )

        return decoder_input

Masked Self-Attention

Decoder looks at:

previous words only
NOT future words

In [9]:



def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)

    # compare query with keys to get attention scores
    scores = torch.matmul(query, key.transpose(-1, -2)) / math.sqrt(d_k)

    # mask hides future tokens in decoder
    if mask is not None:
        scores = scores + mask

    # convert scores into probabilities
    attention_weights = F.softmax(scores, dim=-1)

    # weighted combination of values
    output = torch.matmul(attention_weights, value)

    return output, attention_weights


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, model_dim, num_heads):
        super().__init__()

        assert model_dim % num_heads == 0, \
            "model_dim must be divisible by num_heads"

        self.model_dim = model_dim
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads

        # single projection creates Q, K, and V together
        self.qkv_projection = nn.Linear(model_dim, 3 * model_dim)

        # combines all heads back together
        self.output_projection = nn.Linear(model_dim, model_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # create query, key, value tensors
        qkv = self.qkv_projection(x)

        # split embeddings into multiple heads
        qkv = qkv.reshape(
            batch_size,
            seq_len,
            self.num_heads,
            3 * self.head_dim
        )

        # move heads dimension forward
        qkv = qkv.permute(0, 2, 1, 3)

        # separate query, key, value
        query, key, value = qkv.chunk(3, dim=-1)

        # attention calculation
        context, attention = scaled_dot_product_attention(
            query,
            key,
            value,
            mask
        )

        # combine all heads back
        context = context.reshape(
            batch_size,
            seq_len,
            self.model_dim
        )

        return self.output_projection(context)


Decoder now looks at encoder output.

Basically:

“Which parts of the input sentence matter for this next word?”

In [10]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, model_dim, num_heads):
        super().__init__()

        assert model_dim % num_heads == 0, \
            "model_dim must be divisible by num_heads"

        self.model_dim = model_dim
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads

        # keys and values come from encoder output
        self.kv_projection = nn.Linear(model_dim, 2 * model_dim)

        # queries come from decoder input
        self.q_projection = nn.Linear(model_dim, model_dim)

        self.output_projection = nn.Linear(model_dim, model_dim)

    def forward(self, encoder_output, decoder_input, mask=None):
        batch_size, seq_len, _ = encoder_output.size()

        # create keys and values from encoder output
        kv = self.kv_projection(encoder_output)

        # create queries from decoder input
        query = self.q_projection(decoder_input)

        kv = kv.reshape(
            batch_size,
            seq_len,
            self.num_heads,
            2 * self.head_dim
        )

        query = query.reshape(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        kv = kv.permute(0, 2, 1, 3)
        query = query.permute(0, 2, 1, 3)

        # split keys and values
        key, value = kv.chunk(2, dim=-1)

        # decoder attends to encoder output
        context, attention = scaled_dot_product_attention(
            query,
            key,
            value,
            mask
        )

        # merge all heads back
        context = context.reshape(
            batch_size,
            seq_len,
            self.model_dim
        )

        return self.output_projection(context)

Same purpose as encoder:

stabilize training
prevent exploding/vanishing values

In [11]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()

        self.eps = eps

        # learnable scale and shift parameters
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # normalize across feature dimension
        dims = (-1,)

        mean = x.mean(dim=dims, keepdim=True)
        variance = ((x - mean) ** 2).mean(dim=dims, keepdim=True)

        std = torch.sqrt(variance + self.eps)

        normalized_x = (x - mean) / std

        return self.gamma * normalized_x + self.beta

FeedForward Network

After gathering information:

process each token deeper
add non-linearity
refine representation

Same FFN as encoder:

Linear → ReLU → Dropout → Linear

In [12]:

class FeedForwardNetwork(nn.Module):
    def __init__(self, model_dim, hidden_dim, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(model_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, model_dim)

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [14]:
d_model = 512
num_heads = 8
drop_prob = 0.1
batch_size = 30
max_sequence_length = 200
ffn_hidden = 2048
num_layers = 5

x = torch.randn( (batch_size, max_sequence_length, d_model) ) # English sentence positional encoded
y = torch.randn( (batch_size, max_sequence_length, d_model) ) # Bangla sentence positional encoded
mask = torch.full([max_sequence_length, max_sequence_length] , float('-inf'))
mask = torch.triu(mask, diagonal=1)
decoder = TransformerDecoder(d_model, ffn_hidden, num_heads, drop_prob, num_layers)
out = decoder(x, y, mask)

In [17]:
print(out)
print(out.shape)

tensor([[[ 1.4638,  0.1594,  0.9320,  ..., -0.0925, -0.2549,  0.7896],
         [ 1.9507, -1.6974, -0.4080,  ..., -1.7138,  0.5470,  1.3324],
         [-0.7036,  0.2014,  1.5537,  ..., -2.0942,  1.2811,  1.4900],
         ...,
         [ 2.3099, -1.8056, -0.6121,  ...,  1.7157, -0.4710,  1.8862],
         [-0.4543, -0.3175, -0.0852,  ..., -0.8970,  0.1372,  2.0984],
         [ 1.4334, -3.4483, -0.7629,  ..., -1.1327,  0.2141,  2.9628]],

        [[ 0.1585,  1.2042,  2.0547,  ..., -0.4127,  0.5757,  1.3675],
         [ 0.5948,  1.0837, -0.6247,  ..., -1.0073,  0.5056,  1.2003],
         [ 1.0487,  0.9194, -0.1562,  ...,  0.4140,  0.2975, -0.6185],
         ...,
         [-0.0645, -1.3659, -0.6245,  ..., -0.7211, -0.3602,  0.9060],
         [ 0.7001, -3.7529, -0.3659,  ..., -0.6821,  0.9312,  0.0674],
         [-1.7589, -0.9468, -0.9265,  ...,  1.3652, -0.0824,  2.6509]],

        [[-1.5464, -0.4124, -0.4993,  ..., -1.0029,  1.2743,  0.1511],
         [ 0.6460, -0.1186, -1.2399,  ...,  0